# 01 — Data Preprocessing
**Global Temperature Forecasting Using Multimodal Machine Learning**  
Choudhary & Kulkarni (2026), *Climatic Change* (Springer)  

This notebook covers **§3.1** of the paper:
- Fetching all three climate datasets from public APIs
- Cleaning, aligning, and interpolating missing values
- Saving raw and processed CSVs to `data/`

**Datasets:**
| Dataset | Period | Source |
|---------|--------|--------|
| Berkeley Earth Temperature | 1880–2024 | berkeleyearth.org |
| NOAA Mauna Loa CO₂ | 1958–2024 | gml.noaa.gov |
| SILSO Sunspot Number | 1749–2024 | sidc.be |

In [ ]:
# ── Cell 1: Install dependencies (Colab only) ──────────────────────────────
# !pip install -q pandas numpy matplotlib seaborn scikit-learn statsmodels requests tqdm

In [ ]:
# ── Cell 2: Imports ─────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

# Project modules
from preprocessing import (
    load_berkeley_temperature, load_noaa_co2,
    load_silso_sunspots, merge_datasets
)
from utils import set_global_seed, summarise_df

set_global_seed(42)
print('Imports OK')

In [ ]:
# ── Cell 3: Fetch Berkeley Earth temperature anomaly [Rohde & Hausfather 2020]
temp_df = load_berkeley_temperature(use_cache=True)
summarise_df(temp_df, 'Berkeley Earth')
temp_df.tail()

In [ ]:
# ── Cell 4: Fetch NOAA Mauna Loa CO₂ [NOAA GML 2025] ──────────────────────
co2_df = load_noaa_co2(use_cache=True)
summarise_df(co2_df, 'NOAA CO2')
co2_df.tail()

In [ ]:
# ── Cell 5: Fetch SILSO Sunspot Number [Clette & Lefèvre 2015] ─────────────
ssn_df = load_silso_sunspots(use_cache=True)
summarise_df(ssn_df, 'SILSO')
ssn_df.tail()

In [ ]:
# ── Cell 6: Merge all datasets on monthly date index ───────────────────────
df = merge_datasets(temp_df, co2_df, ssn_df, start_year=1880, end_year=2024)
print(f'Merged: {df.shape[0]:,} rows  |  Null %: {df.isnull().mean().max()*100:.3f}%')
df.head(10)

In [ ]:
# ── Cell 7: Quick overview plot — all three signals 1958-2024 ──────────────
fig, axes = plt.subplots(3, 1, figsize=(16, 9), sharex=True)

mask = df['date'].dt.year >= 1958

axes[0].plot(df.loc[mask,'date'], df.loc[mask,'temp_anomaly'], color='#d62728', lw=0.8)
axes[0].fill_between(df.loc[mask,'date'], df.loc[mask,'temp_anomaly'], alpha=0.3, color='#d62728')
axes[0].set_ylabel('Temp Anomaly (°C)')
axes[0].set_title('(a) Berkeley Earth Global Temperature Anomaly', fontweight='bold')
axes[0].axhline(1.5, color='k', ls=':', lw=1.0, label='Paris 1.5°C')
axes[0].legend(fontsize=9)

axes[1].plot(df.loc[mask,'date'], df.loc[mask,'co2_ppm'], color='#2ca02c', lw=1.0)
axes[1].set_ylabel('CO₂ (ppm)')
axes[1].set_title('(b) NOAA Mauna Loa CO₂ Concentration (Keeling Curve)', fontweight='bold')

axes[2].plot(df.loc[mask,'date'], df.loc[mask,'sunspot_number'], color='#ff7f0e', lw=0.6, alpha=0.8)
axes[2].set_ylabel('Sunspot No.')
axes[2].set_title('(c) SILSO International Sunspot Number (Cycles 19–25)', fontweight='bold')
axes[2].set_xlabel('Year')

plt.suptitle('Figure 1 — Multimodal Climate Dataset Overview (1958–2024)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
Path('../results/figures').mkdir(parents=True, exist_ok=True)
plt.savefig('../results/figures/fig1_dataset_overview.png', dpi=300, bbox_inches='tight')
plt.show()
print('Figure 1 saved.')

In [ ]:
# ── Cell 8: Save processed CSVs ────────────────────────────────────────────
out = Path('../data/processed')
out.mkdir(parents=True, exist_ok=True)

df.to_csv(out / 'climate_merged_raw.csv', index=False)
print(f'Saved merged dataset → {out}/climate_merged_raw.csv  ({df.shape})')

# Summary statistics table
stats = df[['temp_anomaly','co2_ppm','sunspot_number']].describe().round(4)
print('\nDescriptive statistics:')
display(stats)